In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=0.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

## Step 1: Draft a Prompt

Start by writing an initial prompt that you want to improve. For this example, we'll use a simple prompt:

In [3]:
question = "What is a process bottleneck?"

prompt = f"""
Please answer the user's question:

{question}
"""

messages = []
add_user_message(messages, prompt)
print(chat(messages))

# Process Bottleneck

A **process bottleneck** is a point in a workflow or system where the flow of work slows down or stops because that stage cannot keep up with the demand placed on it.

## Key Characteristics

- **Constrains throughput** — It limits how much work can be completed overall
- **Creates delays** — Work backs up before the bottleneck and moves slowly through it
- **Reduces efficiency** — Resources before and after the bottleneck may be underutilized

## Common Examples

- A single cashier at a busy store
- A slow approval step in a hiring process
- Limited server capacity on a website
- One person responsible for a critical task

## Why It Matters

Identifying bottlenecks is important because:
- They're often the best place to focus improvement efforts
- Fixing a bottleneck can dramatically increase overall productivity
- Resources spent elsewhere may have minimal impact on total output

## Simple Analogy

Think of a bottleneck like a water bottle—no matter how wide the

## Step 2: Create an Eval Dataset

Your evaluation dataset contains sample inputs that represent the types of questions or requests your prompt will handle in production. The dataset should include questions that will be interpolated into your prompt template.

In [4]:
dataset = [
    "What's 2+2?",
    "How do I make oatmeal?",
    "How far away is the Moon?",
]

## Step 3: Feed Through Claude

Take each question from your dataset and merge it with your prompt template to create complete prompts. Then send each one to Claude to get responses.



In [5]:
def run_prompt(question, template):
    prompt = template.format(question=question)
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

template_v1 = """
Please answer the user's question:

{question}
"""

results_v1 = []
for q in dataset:
    answer = run_prompt(q, template_v1)
    results_v1.append({"question": q, "answer": answer})
    print(q, "->", answer[:100], "\n")

What's 2+2? -> 2 + 2 = 4 

How do I make oatmeal? -> # How to Make Oatmeal

## Basic Stovetop Method

1. **Measure ingredients**
   - 1 cup water or milk 

How far away is the Moon? -> # Distance to the Moon

The Moon is approximately **238,900 miles** (384,400 kilometers) from Earth  



Note: I used `.format()` with a ``{question}`` placeholder, so one template can be reused for every question.

## Step 4: Feed Through a Grader

The grader evaluates the quality of Claude's responses by examining both the original question and Claude's answer. This step provides objective scoring, typically on a scale from 1 to 10, where 10 represents a perfect answer and lower scores indicate room for improvement.

In [7]:
def grade(question, answer):
    grader_prompt = f"""
You are a strict grader. Evaluate the answer to the question.

Penalise heavily (score 5 or lower) if the answer:
- is incomplete or cut off
- omits quantities, steps or times needed to actually do the task
- is vague, generic or padded
- does not directly answer what was asked

Give 9-10 only for complete, specific, accurate answers.

Question: {question}

Answer: {answer}

First write two short sentences of reasoning, then on a new line write
"Score: " followed by a number from 1 to 10.
"""
    messages = []
    add_user_message(messages, grader_prompt)
    reply = chat(messages, temperature=0.0)
    return int(reply.split("Score:")[-1].strip().split()[0])

def evaluate(results):
    scores = []
    for r in results:
        s = grade(r["question"], r["answer"])
        scores.append(s)
        print(s, "-", r["question"])
    avg = sum(scores) / len(scores)
    print("Average:", round(avg, 2))
    return avg

score_v1 = evaluate(results_v1)

10 - What's 2+2?
9 - How do I make oatmeal?
9 - How far away is the Moon?
Average: 9.33


In [8]:
def grade_debug(question, answer):
    grader_prompt = f"""
You are a strict grader. Evaluate the answer to the question.

Penalise heavily (score 5 or lower) if the answer:
- is incomplete or cut off
- omits quantities, steps or times needed to actually do the task
- is vague, generic or padded
- does not directly answer what was asked

Give 9-10 only for complete, specific, accurate answers.

Question: {question}

Answer: {answer}

First write two short sentences of reasoning, then on a new line write
"Score: " followed by a number from 1 to 10.
"""
    messages = []
    add_user_message(messages, grader_prompt)
    return chat(messages, temperature=0.0)

print(grade_debug(results_v1[1]["question"], results_v1[1]["answer"]))

This answer directly addresses the question with specific, actionable instructions including exact quantities, cooking times, and temperatures. It provides two practical methods (stovetop and microwave) with clear step-by-step directions, plus helpful tips and topping suggestions that enhance the answer without padding it.

Score: 9


In [9]:
print(results_v1[1]["answer"])

# How to Make Oatmeal

## Basic Stovetop Method

1. **Measure ingredients**
   - 1 cup water or milk (or a mix)
   - ½ cup rolled oats
   - Pinch of salt

2. **Bring liquid to a boil**

3. **Add oats and salt**
   - Stir to combine

4. **Reduce heat and simmer**
   - Cook for 5 minutes, stirring occasionally
   - Add more liquid if it's too thick

5. **Serve**
   - Transfer to a bowl
   - Add toppings of your choice

## Quick Microwave Method

1. Mix ½ cup oats, 1 cup water/milk, and salt in a microwave-safe bowl
2. Microwave on high for 2-3 minutes
3. Stir and let cool for a minute
4. Add toppings and serve

## Popular Toppings

- Fresh fruit (berries, bananas, apples)
- Nuts and seeds
- Honey or maple syrup
- Cinnamon or vanilla
- Yogurt
- Chocolate chips

**Tip:** Use milk or a milk alternative for creamier oatmeal, and adjust cooking time based on whether you prefer thicker or thinner consistency.


## Step 5: Change Prompt and Repeat

Now that you have a baseline score, you can modify your prompt and run the entire process again to see if your changes improve performance.

In [10]:
template_v2 = """
Please answer the user's question:

{question}

Answer the question with ample detail
"""

results_v2 = [{"question": q, "answer": run_prompt(q, template_v2)} for q in dataset]
score_v2 = evaluate(results_v2)

print("v1:", round(score_v1, 2), "| v2:", round(score_v2, 2))

7 - What's 2+2?
9 - How do I make oatmeal?
9 - How far away is the Moon?
Average: 8.33
v1: 9.33 | v2: 8.33


In [11]:
print(results_v1[0]["answer"])

2 + 2 = 4


In [12]:
print(results_v2[0]["answer"])

# What's 2+2?

**The answer is 4.**

## Explanation

When we add 2 and 2 together, we're combining two groups of two items each:

- First group: ••
- Second group: ••
- Combined: •••• (four items total)

## Mathematical Representation

This can be written as:
- **2 + 2 = 4**

## Why This Works

Addition is one of the fundamental operations in arithmetic. When you add two numbers, you're finding the total when combining quantities. In this case:
- Start with 2
- Add 2 more
- You end up with 4

This is one of the most basic arithmetic facts and forms the foundation for more complex mathematics. It's typically one of the first equations children learn in elementary mathematics.


In [13]:
template_v3 = """
Please answer the user's question:

{question}

Match the amount of detail to the question: be brief for simple questions
and thorough for complex ones.
"""

results_v3 = [{"question": q, "answer": run_prompt(q, template_v3)} for q in dataset]
score_v3 = evaluate(results_v3)
print("v1:", round(score_v1, 2), "| v2:", round(score_v2, 2), "| v3:", round(score_v3, 2))

10 - What's 2+2?
8 - How do I make oatmeal?
9 - How far away is the Moon?
Average: 9.0
v1: 9.33 | v2: 8.33 | v3: 9.0
